# Get corekit from GitHub

coregit is a library of useful utilities that we can utilize for generation and tuning

In [1]:
import os
# os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
import sys
sys.path.append('/raid_storage/SLURM/home/slurm_majedalshaibani/Projects/instructions-tuning/jrcai_corekit/llms_corekit')

check everything is working

# Constants

In [4]:
TAWJEEH_DATASET_NAME = 'ArSarcasm_v2'
HF_EXPERIMENTAL_DATASET_NAME = 'MagedSaeed/ArSarcasm_v2_experimental'
MODEL_PATH = "/raid_storage/shared_models/Qwen3-8B-Base"
MODEL_NAME = "Qwen3-8B"
TASK_NAME='sarcasm_detection'

In [5]:
TOKENIZER_PATH = MODEL_PATH

# Building the prompts dataset

In [6]:
import requests

from tqdm.auto import tqdm

prompts = None

tries = 10
for i in tqdm(range(tries)):
    api_response = requests.get(url='https://promptlab.up.railway.app/api/prompt/list?project_secret_key=6Wirj')
    if api_response.ok:
        prompts = api_response.json()
        break
if not prompts: raise Exception('Failed to fetch prompts')
prompts[:5]

  0%|          | 0/10 [00:00<?, ?it/s]

[{'id': 14901,
  'tags': [],
  'name': 'A Simple Test Prompt',
  'task': {'name': 'dialect identification'},
  'status': 'DRAFT',
  'template': 'Please predict the most suitable dialect for the following text: {{arabic}}\xa0\r\n|||{{answer_choices[label]}}',
  'created_by': 'irfan',
  'dataset_name': 'arbml/AraBench_dev',
  'dataset_subset': 'default',
  'answer_choices': ['Tunisian',
   'MSA',
   'Morrocan',
   'Qatari',
   'Egyptian',
   'Lebanese'],
  'text_direction': 'ltr'},
 {'id': 14898,
  'tags': ['', 'Zero-shot COT'],
  'name': 'Prompt with zero-shot chain of thoughts',
  'task': {'name': 'claim verification'},
  'status': 'APPROVED',
  'template': "For the following task you have to label if the two sentences are of on of the following labels: {% for choice in answer_choices %}{{ choice }}{% if not loop.last %} or {% endif %}{% endfor %}. Sentence 1: {{s1}}\xa0 and sentence 2: {{s2}}.\r\nLet's think step by step:\r\n|||\r\n{{answer_choices[label]}}",
  'created_by': 'ahmed',


In [7]:
len(prompts)

365

In [8]:
filtered_prompts = list(filter(lambda prompt: prompt['status'] == 'APPROVED' and prompt['text_direction'].lower() == 'ltr', prompts))
len(filtered_prompts)

352

## Finetuning

### Get the dataset prompts

In [9]:
# you can either filter by task or dataset
dataset_prompts = list(
    filter(
        lambda prompt: TAWJEEH_DATASET_NAME in prompt['dataset_name'],
        filtered_prompts,
    )
)
len(dataset_prompts)

8

In [10]:
SELECTED_PROMPTS_IDS = [
    14779,
    14802,
    14835,
    14837,
    14838,
]

In [11]:
dataset_prompts = list(filter(lambda prompt: prompt['id'] in SELECTED_PROMPTS_IDS, filtered_prompts))
len(dataset_prompts)

5

### Download the dataset

In [12]:
import datasets

In [13]:
hf_exp_dataset = datasets.load_dataset(HF_EXPERIMENTAL_DATASET_NAME)
hf_exp_dataset

DatasetDict({
    train: Dataset({
        features: ['tweet', 'sentiment', 'dialect', 'label'],
        num_rows: 12548
    })
    test: Dataset({
        features: ['tweet', 'sentiment', 'dialect', 'label'],
        num_rows: 3000
    })
})

### Merge the prompts

In [14]:
from jinja2 import Environment, StrictUndefined

In [15]:
import re
def preprocess_template(template):
    # remove punc at the end
    prefix,suffix = template.split('|||')
    # remove multi spaces
    # prefix = re.sub(r'\s+', ' ', prefix)
    return f'{prefix.strip()}\n{suffix.strip()}' # output is always the last line!

In [16]:
def apply_template(prompt_template, sample):
    try:
        template = prompt_template['template']
        template = preprocess_template(template)
        sample['answer_choices'] = prompt_template['answer_choices']
        env = Environment(undefined=StrictUndefined)
        template = env.from_string(template)
        rendered_template = template.render(**sample)
        return rendered_template
    except Exception as e:
        print(prompt_template)
        print(sample)
        raise e

### Perform prompt-merge on one example prompt, for experimentation

In [17]:
example_prompt_template = dataset_prompts[0]
print(apply_template(example_prompt_template, hf_exp_dataset['train'][3]))

Task: Identify whether the following tweet is sarcastic or non-sarcastic by comparing it to typical sarcasm cues.

Indicators of Sarcasm:
- Exaggerated praise or criticism
- Statements that imply the opposite of their literal meaning
- Use of irony or unexpected humor

Tweet: "صراع شرس بين ريال مدريد و برشلونة على ثنائي اتليتكو مدريدالتفاصيل: https://t.co/ZQHObvGEnP#إبداع_الملاعب https://t.co/sCCW1u5LBN"

Answer: Based on the indicators, respond with "sarcastic" if the tweet shows sarcasm, or "non-sarcastic" if it does not.
Non sarcastic


In [18]:
step_size = int(len(hf_exp_dataset['train'])/len(dataset_prompts))
step_size

2509

In [19]:
rendered_train_prompts_dataset = list()
for i,sample in enumerate(tqdm(hf_exp_dataset['train'])):
    if i % step_size == 0:
        print(f'rending {dataset_prompts[min(int(i/step_size), len(dataset_prompts)-1)]["template"]}','sample index:',i)
    rendered_train_prompts_dataset.append(
        apply_template(dataset_prompts[min(int(i/step_size), len(dataset_prompts)-1)], sample)
    )
len(rendered_train_prompts_dataset)

  0%|          | 0/12548 [00:00<?, ?it/s]

rending Task: Identify whether the following tweet is sarcastic or non-sarcastic by comparing it to typical sarcasm cues.

Indicators of Sarcasm:
- Exaggerated praise or criticism
- Statements that imply the opposite of their literal meaning
- Use of irony or unexpected humor

Tweet: {{ tweet }}

Answer: Based on the indicators, respond with "sarcastic" if the tweet shows sarcasm, or "non-sarcastic" if it does not.
|||
{{ answer_choices[label] }} sample index: 0


rending Task: Determine if the following tweet contains sarcasm by following these steps.

Steps:
1. Understand the Content: Read the tweet carefully to grasp the main idea or message.
2. Look for Sarcastic Cues: Consider if there are exaggeration, irony, or statements that may imply a hidden meaning.
3. Make a Decision: Based on the cues, decide if the tweet is sarcastic or non-sarcastic.

Tweet: {{ tweet }}

Answer: Based on the previous steps, respond with only one of these options (sarcastic or non-sarcastic). Do not include any additional explanation.
|||
{{ answer_choices[label] }} sample index: 2509


rending Task: Determine if the following tweet is sarcastic or non-sarcastic.

Tweet: {{ tweet }}

Answer: Respond with "sarcastic" if the tweet is sarcastic, or "non-sarcastic" if it is not. No need for extra explanation.
|||
{{ answer_choices[label] }} sample index: 5018


rending The following tweet: {{tweet}} is sarcastic? 
{{answer_choices | join(' or ')}}
|||
{{answer_choices[not label]}} sample index: 7527


rending Sarcasm is a form of verbal irony that is intended to express contempt or ridicule. Given the following tweet: {{tweet}}, predict if it is 'Sarcastic" or "Non Sarcastic".
|||
{{answer_choices[label]}} sample index: 10036


rending Sarcasm is a form of verbal irony that is intended to express contempt or ridicule. Given the following tweet: {{tweet}}, predict if it is 'Sarcastic" or "Non Sarcastic".
|||
{{answer_choices[label]}} sample index: 12545


12548

## Finetune the LLM

In [20]:
GLOBAL_SEED = 42

In [21]:
import random
random.seed(GLOBAL_SEED)

In [22]:
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from llm import train_llm, LLMLoader, Qwen3Initializer, LoRAConfigRepository
from sklearn.model_selection import train_test_split

🚨 Config not found for parakeet. You can manually add it to HARDCODED_CONFIG_FOR_MODELS in utils/auto_docstring.py
🚨 Config not found for parakeet. You can manually add it to HARDCODED_CONFIG_FOR_MODELS in utils/auto_docstring.py
🚨 Config not found for parakeet. You can manually add it to HARDCODED_CONFIG_FOR_MODELS in utils/auto_docstring.py


/raid_storage/SLURM/home/slurm_majedalshaibani/Projects/instructions-tuning/jrcai_corekit/llms_corekit/llm/message_generator.py:5: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [23]:
llm_loader = LLMLoader(
        MODEL_PATH,
        llm_initializer=Qwen3Initializer(),
)
llm_loader

In [24]:
model, tokenizer, generation_config = llm_loader()

loading configuration file /raid_storage/shared_models/Qwen3-8B-Base/config.json


`torch_dtype` is deprecated! Use `dtype` instead!


Model config Qwen3Config {
  "architectures": [
    "Qwen3ForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151643,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 12288,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_at

loading weights file /raid_storage/shared_models/Qwen3-8B-Base/model.safetensors.index.json


Instantiating Qwen3ForCausalLM model under default dtype torch.bfloat16.


...

loading file tokenizer_config.json


loading file chat_template.jinja


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


loading configuration file /raid_storage/shared_models/Qwen3-8B-Base/generation_config.json


Generate config GenerationConfig {
  "bos_token_id": 151643,
  "eos_token_id": 151643,
  "max_new_tokens": 2048
}



In [25]:
import re

train_samples,eval_samples = train_test_split(
    rendered_train_prompts_dataset,
    test_size=0.1,
    random_state=GLOBAL_SEED,
)

def generate_tuple(sample):
    sample_lines = sample.splitlines()
    prefix = '\n'.join(sample_lines[:-1])
    prefix = prefix.strip()
    suffix = sample_lines[-1].strip()
    suffix = f' {suffix}' # adding this space is important to split between input and output
    return prefix,suffix

train_samples = list(map(generate_tuple,train_samples))
eval_samples = list(map(generate_tuple,eval_samples))
len(train_samples), len(eval_samples), train_samples[:5], eval_samples[:5]

(11293,
 1255,
 [('Task: Determine if the following tweet is sarcastic or non-sarcastic.\n\nTweet: بالفيديو عضو بحملته الانتخابية: #السيسي لا يقابلنا لأنه "مرشح غير عادي" ويعشق لقب #المشير ولا يزال متمسكا به\n\nAnswer: Respond with "sarcastic" if the tweet is sarcastic, or "non-sarcastic" if it is not. No need for extra explanation.',
   ' Non sarcastic'),
  ('Task: Identify whether the following tweet is sarcastic or non-sarcastic by comparing it to typical sarcasm cues.\n\nIndicators of Sarcasm:\n- Exaggerated praise or criticism\n- Statements that imply the opposite of their literal meaning\n- Use of irony or unexpected humor\n\nTweet: 99% من منتقدين هاري بوتر ما شافوه وال %1 مجرد حشرات لا يستطيع اي شي تطهيرها\n\nAnswer: Based on the indicators, respond with "sarcastic" if the tweet shows sarcasm, or "non-sarcastic" if it does not.',
   ' sarcastic'),
  ('Sarcasm is a form of verbal irony that is intended to express contempt or ridicule. Given the following tweet: "@roose9111 امريكا

In [26]:
train_llm(
    model=model,
    tokenizer=tokenizer,
    train_samples=train_samples,
    eval_samples=eval_samples,
    peft_config=LoRAConfigRepository.llama_3(),
    learning_rate=2.5e-4,
    epochs_count=10,
    train_batch_size=16,
    eval_batch_size=16,
    output_dir=f'Notebooks/Experiments/{TASK_NAME}/{TAWJEEH_DATASET_NAME}/tuned_models/{MODEL_NAME}'
)

PyTorch: setting up devices


The default value for the training argument `--report_to` will change in v5 (from all installed integrations to none). In v5, you will need to use `--report_to all` to get the same behavior as now. You should start updating your code and make this info disappear :-).


/raid_storage/SLURM/home/slurm_majedalshaibani/Projects/instructions-tuning/jrcai_corekit/llms_corekit/llm/llm_trainer.py:83: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
You have loaded a model on multiple GPUs. `is_model_parallel` attribute will be force-set to `True` to avoid any unexpected behavior such as device placement mismatching.


Using auto half precision backend



***** Running Evaluation *****


...

  Num examples = 1255


  Batch size = 16


{'eval_loss': 0.10469332337379456, 'eval_model_preparation_time': 0.0002, 'eval_runtime': 22.9044, 'eval_samples_per_second': 54.793, 'eval_steps_per_second': 3.449, 'epoch': 2.8328611898017}




Training completed. Do not forget to share your model on huggingface.co/models =)




0.08921197801828384

In [27]:
exit()